# Computational Control: Tutorial 1 (solutions)
**Convex optimization with CVXPY (short version)**

In this tutorial we learn how to set up and solve convex optimization problems in Python with [**CVXPY**](https://www.cvxpy.org/), an open-source modelling language. We proceed in three steps:
1. a demo on a problem small enough to solve by hand, first without and then with constraints;
2. an exercise where you write a problem from scratch;
3. an exercise where you plan the motion of a robot and watch it move.

<font color="#40B0A6"><b>🧭 NEW TO MARKDOWN OR GOOGLE COLAB?</b></font> Start with Tutorial 0 (`COCO_T0_Colab`), posted alongside this notebook.

<font color="#40B0A6"><b>🎯 WHY THIS MATTERS</b></font>

Every controller in this course is computed by solving an optimization problem. The linear quadratic regulator (LQR) minimizes a quadratic cost over input sequences; model predictive control (MPC) solves a constrained convex problem at every time step; system identification fits models to data by convex optimization; and data-driven predictive control uses recorded data directly inside a convex problem. CVXPY lets us write these problems in code almost exactly as we write them on the board.

<font color="#009E73"><b>✅ SOLUTIONS</b></font>

This version contains the solutions of all exercises and the answers to the 🤔 PREDICT FIRST questions.

<font color="#E69F00"><b>⚠ ATTENTION: BEFORE YOU START</b></font>

This notebook is shared read-only. Choose **File → Save a copy in Drive** and work on your copy, otherwise your edits are lost.
Run a cell with **Shift+Enter**; **Runtime → Run all** runs the whole notebook from the top.

### Package imports

In [1]:
# Package imports
import numpy as np                   # arrays and linear algebra
import cvxpy as cp                   # convex optimization modelling language (preinstalled on Colab)
import plotly.graph_objects as go    # interactive plots
import plotly.io as pio

# Print NumPy arrays with 3 decimals, without scientific notation for tiny numbers
np.set_printoptions(precision=3, suppress=True)

# Plot style: white background, tight margins
pio.templates["coco"] = go.layout.Template(
    layout=dict(autosize=True, margin=dict(l=30, r=30, t=30, b=30))
)
pio.templates.default = "plotly_white+coco"

### Convex optimization problems in CVXPY

A *convex optimization problem* has the form
$$
\begin{array}{ll}
\underset{x \in \mathbb{R}^n}{\text{minimize}} & f(x) \\
\text{subject to} & g_i(x) \le 0, \quad i = 1, \dots, n_g, \\
& h_j(x) = 0, \quad j = 1, \dots, n_h,
\end{array}
$$
where the cost $f$ and the constraint functions $g_i$ are convex and the $h_j$ are affine.
A function $f$ is *convex* if $f(s x + (1-s) z) \le s f(x) + (1-s) f(z)$ for all $x, z$ and all $s \in [0, 1]$: every chord of its graph lies on or above the graph. A function is *affine* if it is linear plus a constant.
For convex problems every local minimizer is a global minimizer, and solvers find one reliably.

In CVXPY, every problem is written in five steps:

| step | code |
|---|---|
| 1. decision variables | `x = cp.Variable(n)` |
| 2. cost | `cost = ...`, an expression in `x` |
| 3. constraints | `constraints = [...]`, a list of relations built with `<=`, `>=`, `==` |
| 4. problem | `problem = cp.Problem(cp.Minimize(cost), constraints)` |
| 5. solve | `problem.solve()`, then read `x.value`, `problem.value`, `problem.status` |

CVXPY checks convexity with a set of composition rules called [disciplined convex programming](https://www.cvxpy.org/tutorial/dcp/index.html) (DCP). If it cannot certify that a problem is convex, it refuses to solve it and raises a `DCPError`.

In [2]:
# Plotting helper: run this cell, no need to read it.
def plot_problem(f, vertices=None, points=(), xlim=(-1, 3), ylim=(-1, 3)):
    # Contour lines of f(x1, x2), the feasible polygon with the given vertices, and labelled points
    x1 = np.linspace(*xlim, 200)
    x2 = np.linspace(*ylim, 200)
    X1, X2 = np.meshgrid(x1, x2)
    fig = go.Figure(go.Contour(x=x1, y=x2, z=f(X1, X2), ncontours=20, showscale=False,
                               contours_coloring="lines", colorscale=[[0, "gray"], [1, "gray"]], line_width=1,
                               name="level sets of f"))
    if vertices is not None:
        V = np.vstack([vertices, vertices[:1]])
        fig.add_trace(go.Scatter(x=V[:, 0], y=V[:, 1], mode="lines", fill="toself",
                                 fillcolor="rgba(86, 180, 233, 0.3)", line=dict(color="#56B4E9"),
                                 name="feasible set"))
    for label, pt, color in points:
        fig.add_trace(go.Scatter(x=[pt[0]], y=[pt[1]], mode="markers", name=label,
                                 marker=dict(size=12, color=color, line=dict(width=1, color="black"))))
    fig.update_layout(width=560, height=500, xaxis_title="x_1", yaxis_title="x_2",
                      xaxis=dict(range=xlim), yaxis=dict(range=ylim, scaleanchor="x", scaleratio=1))
    fig.show()

### A problem we can solve by hand

<font color="#40B0A6"><b>▶️ DEMO</b></font>

Consider the cost $f(x) = (x_1 - 1.5)^2 + (x_2 - 2)^2$, the squared distance from $x$ to the point $c = (1.5, 2)$.
1. Without constraints, the minimizer is $x^\star = c$ and the optimal value is $p^\star = 0$.
2. We then add the constraints $x_1 \ge 0$, $x_2 \ge 0$, $x_1 + x_2 \le 1$, which define a triangle:
$$
\begin{array}{ll}
\underset{x \in \mathbb{R}^2}{\text{minimize}} & (x_1 - 1.5)^2 + (x_2 - 2)^2 \\
\text{subject to} & x_1 \ge 0, \quad x_2 \ge 0, \quad x_1 + x_2 \le 1.
\end{array}
$$

<font color="#A890FF"><b>🤔 PREDICT FIRST</b></font>

Since $f$ is a squared distance, the constrained minimizer is the point of the triangle closest to $c$. Sketch it: where is it, and which constraints hold with equality there?

<font color="#CC79A7"><b>✨ SOLUTION</b></font>

The closest point lies on the edge $x_1 + x_2 = 1$. Projecting $c$ onto the line $x_1 + x_2 = 1$ gives $c - \tfrac{1.5 + 2 - 1}{2}\,(1, 1) = (0.25, 0.75)$, which lies in the triangle. Hence $x^\star = (0.25, 0.75)$ and $p^\star = 2 \cdot 1.25^2 = 3.125$. Only the constraint $x_1 + x_2 \le 1$ holds with equality.

In [3]:
# --- 1. Without constraints ---
x = cp.Variable(2)                                    # step 1: decision variable x in R^2
cost = (x[0] - 1.5)**2 + (x[1] - 2)**2                # step 2: cost
problem = cp.Problem(cp.Minimize(cost))               # steps 3-4: no constraints
problem.solve()                                       # step 5: solve
print("Without constraints:  status =", problem.status, "  x* =", x.value, f"  p* = {problem.value:.4f}")
x_unc = x.value.copy()

# --- 2. With constraints ---
constraints = [x[0] >= 0,                             # step 3: constraints
               x[1] >= 0,
               x[0] + x[1] <= 1]
problem = cp.Problem(cp.Minimize(cost), constraints)  # step 4
problem.solve()                                       # step 5
print("With constraints:     status =", problem.status, "  x* =", x.value, f"  p* = {problem.value:.4f}")
x_con = x.value.copy()

# Each constraint also carries a Lagrange multiplier (its "dual value"): a number that measures
# how strongly the constraint pushes on the solution. It is zero for constraints that are not active,
# i.e. that do not hold with equality at x*.
print("Multipliers:", [round(float(con.dual_value), 4) for con in constraints])

Without constraints:  status = optimal   x* = [1.5 2. ]   p* = 0.0000
With constraints:     status = optimal   x* = [0.25 0.75]   p* = 3.1250
Multipliers: [0.0, 0.0, 2.5]


In [4]:
plot_problem(lambda x1, x2: (x1 - 1.5)**2 + (x2 - 2)**2,
             vertices=np.array([[0., 0.], [1., 0.], [0., 1.]]),
             points=[("minimizer without constraints", x_unc, "white"),
                     ("minimizer with constraints", x_con, "coral")],
             xlim=(-0.5, 2.5), ylim=(-0.5, 2.5))

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

The constrained minimizer is the point where a level set of $f$ first touches the triangle. There, the level set is tangent to the active edge.

### Exercise 1: a convex program from scratch

<font color="#A890FF"><b>✏️ EXERCISE</b></font>

Solve
$$
\begin{array}{ll}
\underset{x \in \mathbb{R}^2}{\text{minimize}} & x_1^2 + (x_2 - 2)^2 \\
\text{subject to} & x_1 + x_2 \le 1, \quad x_1 \ge -1, \quad x_2 \ge -1.
\end{array}
$$
Store the minimizer in `x_star` and the optimal value in `opt_value`. The cell plots your solution.

The three constraints can be written at once as $G x \le h$, entrywise, with
$$
G = \begin{bmatrix} 1 & 1 \\ -1 & 0 \\ 0 & -1 \end{bmatrix}, \qquad h = \begin{bmatrix} 1 \\ 1 \\ 1 \end{bmatrix}.
$$
Constraints of this form are called *polyhedral*. They are how input and state limits are written in MPC.

<font color="#A890FF"><b>🤔 PREDICT FIRST</b></font>

Before coding: which point will the solver return, and what is $p^\star$? Then change $h_1$ from $1$ to $3$ and predict again.

<font color="#CC79A7"><b>✨ SOLUTION</b></font>

The cost is the squared distance to $(0, 2)$, a point that violates $x_1 + x_2 \le 1$. Projecting $(0, 2)$ onto the line $x_1 + x_2 = 1$ gives $(0, 2) - \tfrac{0 + 2 - 1}{2}\,(1, 1) = (-0.5, 1.5)$, which satisfies the other two constraints. Hence $x^\star = (-0.5, 1.5)$ and $p^\star = 0.25 + 0.25 = 0.5$. With $h_1 = 3$ the point $(0, 2)$ is feasible, so $x^\star = (0, 2)$, $p^\star = 0$, and no constraint is active.

<font color="#40B0A6"><b>💡 HINT</b></font>

A constraint can be a vector inequality such as `G @ x <= h`, which holds entrywise. For the other steps, look back at the demo.

In [ ]:
# Problem data (from the text)
G = np.array([[ 1.,  1.],
              [-1.,  0.],
              [ 0., -1.]])
h = np.array([1., 1., 1.])        # try h[0] = 3

############################ Your code goes here ###############################
x = cp.Variable(2)
cost = cp.sum_squares(x - np.array([0., 2.]))
constraints = [G @ x <= h]
problem = cp.Problem(cp.Minimize(cost), constraints)
problem.solve()

x_star = x.value
opt_value = problem.value
################################################################################

assert x_star is not None and opt_value is not None, "Fill in the TODO lines in the block above first."
print("x* =", x_star, f"  p* = {opt_value:.4f}")

# The feasible set is the triangle with vertices (-h2, -h3), (h1 + h3, -h3), (-h2, h1 + h2)
plot_problem(lambda x1, x2: x1**2 + (x2 - 2)**2,
             vertices=np.array([[-h[1], -h[2]], [h[0] + h[2], -h[2]], [-h[1], h[0] + h[1]]]),
             points=[("your minimizer", x_star, "coral")],
             xlim=(-1.5, 2.5), ylim=(-1.5, 2.5))

### Exercise 2: convex optimization in action, planning a robot's motion

A robot moves in the plane. At time $t$ its position is $p_t \in \mathbb{R}^2$, its velocity is $v_t \in \mathbb{R}^2$, and we choose its acceleration $u_t \in \mathbb{R}^2$, the *input*. With time step $\Delta t$, a simple (forward Euler) discretization of Newton's law gives
$$
p_{t+1} = p_t + \Delta t\, v_t, \qquad v_{t+1} = v_t + \Delta t\, u_t, \qquad t = 0, 1, \dots, N-1.
$$
The robot starts at rest at $p_{\text{start}} = (0, 0)$ and must arrive at rest at $p_{\text{goal}} = (10, 0)$ at time $N$. A wall at first coordinate $5$ has a door where the second coordinate is between $2$ and $4$, so at time $t_{\text{gate}}$ the robot must be in the doorway, with a safety margin. The motors are limited: $|u_{t,i}| \le u_{\max}$. Among all input sequences that do the job, we look for one with the least *control effort* $\sum_t \|u_t\|_2^2$:
$$
\begin{array}{ll}
\underset{p,\, v,\, u}{\text{minimize}} & \sum_{t=0}^{N-1} \|u_t\|_2^2 \\
\text{subject to} & p_{t+1} = p_t + \Delta t\, v_t, \quad v_{t+1} = v_t + \Delta t\, u_t, \quad t = 0, \dots, N-1, \\
& p_0 = p_{\text{start}}, \quad v_0 = 0, \quad p_N = p_{\text{goal}}, \quad v_N = 0, \\
& |u_{t,i}| \le u_{\max}, \quad t = 0, \dots, N-1, \quad i = 1, 2, \\
& p_{t_{\text{gate}},1} = 5, \quad 2.5 \le p_{t_{\text{gate}},2} \le 3.5 .
\end{array}
$$
The cost is a convex quadratic and every constraint is a linear equality or inequality ($|u_{t,i}| \le u_{\max}$ is the pair $-u_{\max} \le u_{t,i} \le u_{\max}$), so this is a convex problem, a *quadratic program*.

Fixing the time $t_{\text{gate}}$ at which the robot crosses the wall keeps the problem convex. Letting the optimizer also choose the crossing time, or the side on which to pass an obstacle, would make it non-convex.

<font color="#40B0A6"><b>🎯 WHY THIS MATTERS</b></font>

This is the problem at the heart of model predictive control (MPC): use a model to predict the effect of future inputs, and optimize those inputs subject to constraints. MPC solves a problem like this one at every time step, starting from the current measured state (Tutorial 4). The two update equations form a *state-space model*, the topic of Tutorial 2.

---

<font color="#A890FF"><b>✏️ EXERCISE</b></font>

The variables and the dynamics constraints are given. Complete the program:
1. the cost (control effort);
2. the start and goal constraints;
3. the input bounds;
4. the gate constraint.

Then solve, and run the animation cell below.

<font color="#A890FF"><b>🤔 PREDICT FIRST</b></font>

- (a) Without the gate constraint, which path would the robot take?
- (b) With a large $u_{\max}$ the optimal inputs stay below $0.63$ in absolute value. What happens with $u_{\max} = 0.5$? With $u_{\max} = 0.4$? Try it.

<font color="#CC79A7"><b>✨ SOLUTION</b></font>

- (a) A straight line along the first coordinate axis: the motion in the second coordinate starts and ends at rest at $0$, and any nonzero $u_{t,2}$ only adds cost.
- (b) With $u_{\max} = 0.5$ the bound becomes active: the inputs saturate at $\pm 0.5$ for some time steps, and the control effort grows slightly (from about $9.63$ to $9.74$). With $u_{\max} = 0.4$ the robot cannot reach the doorway and the goal in time: `problem.status` is `infeasible` and there is nothing to animate. The smallest feasible bound is about $0.43$.

<font color="#40B0A6"><b>💡 HINT</b></font>

Slicing works on CVXPY variables as on NumPy arrays: `p[:, 0]` is $p_0$ and `p[1, t_gate]` is the second entry of $p_{t_{\text{gate}}}$. `cp.abs(u) <= u_max` bounds all entries at once, and `cp.sum_squares(u)` adds up the squares of all entries. Append constraints with `constraints += [...]`.

In [ ]:
# --- Problem data ---
N = 40                            # horizon: number of time steps
dt = 0.25                         # time step [s]
p_start = np.array([0., 0.])      # start position
p_goal = np.array([10., 0.])      # goal position
t_gate = N // 2                   # time step at which the robot must be in the doorway
gate_x, gate_y = 5.0, (2.5, 3.5)  # doorway: first coordinate 5, second coordinate between 2.5 and 3.5
u_max = 1.0                       # input bound (try 0.5 and 0.4)

# --- Decision variables, one column per time step ---
p = cp.Variable((2, N + 1))       # positions  p_0, ..., p_N
v = cp.Variable((2, N + 1))       # velocities v_0, ..., v_N
u = cp.Variable((2, N))           # inputs     u_0, ..., u_{N-1}

# --- Dynamics (given) ---
constraints = []
for t in range(N):
    constraints += [p[:, t + 1] == p[:, t] + dt * v[:, t],
                    v[:, t + 1] == v[:, t] + dt * u[:, t]]

############################ Your code goes here ###############################
cost = cp.sum_squares(u)
constraints += [p[:, 0] == p_start, v[:, 0] == 0,
                p[:, N] == p_goal,  v[:, N] == 0]
constraints += [cp.abs(u) <= u_max]
constraints += [p[0, t_gate] == gate_x,
                p[1, t_gate] >= gate_y[0],
                p[1, t_gate] <= gate_y[1]]
problem = cp.Problem(cp.Minimize(cost), constraints)
problem.solve()
################################################################################

assert problem is not None, "Fill in the TODO lines in the block above first."
print("status:", problem.status)
if u.value is not None:
    print(f"control effort = {problem.value:.3f},   max |u| = {np.max(np.abs(u.value)):.3f}")

Run the next cell and press **▶ Play**.

In [ ]:
# --- Watch the robot move ---
status = getattr(problem, "status", None)
assert status in ("optimal", "optimal_inaccurate"), f"Nothing to animate: problem status is {status}."
P = p.value
wall = dict(color="black", width=5)
door_lo, door_hi = gate_y[0] - 0.5, gate_y[1] + 0.5   # physical doorway, wider than the gate constraint
y_top = max(6.0, door_hi + 1.0)

fig = go.Figure(
    data=[go.Scatter(x=[gate_x, gate_x], y=[-3, door_lo], mode="lines", line=wall, name="wall"),
          go.Scatter(x=[gate_x, gate_x], y=[door_hi, y_top], mode="lines", line=wall, showlegend=False),
          go.Scatter(x=P[0], y=P[1], mode="lines", line=dict(color="gray", dash="dot"), name="planned path"),
          go.Scatter(x=[p_start[0], p_goal[0]], y=[p_start[1], p_goal[1]], mode="markers",
                     marker=dict(symbol="x", size=10, color="black"), name="start and goal"),
          go.Scatter(x=[P[0, 0]], y=[P[1, 0]], mode="markers", name="robot",
                     marker=dict(size=20, color="coral", line=dict(width=1, color="black")))],
    frames=[go.Frame(data=[go.Scatter(x=[P[0, t]], y=[P[1, t]])], traces=[4], name=str(t))
            for t in range(N + 1)],
)
fig.update_layout(
    height=450, xaxis=dict(range=[-1, 11], title="position, first coordinate"),
    yaxis=dict(range=[-3, y_top], title="position, second coordinate", scaleanchor="x", scaleratio=1),
    updatemenus=[dict(type="buttons", x=0, y=1.15, xanchor="left", direction="right", buttons=[
        dict(label="▶ Play", method="animate",
             args=[None, dict(frame=dict(duration=1000 * dt, redraw=False), fromcurrent=True,
                              transition=dict(duration=0))]),
        dict(label="⏸ Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(currentvalue=dict(prefix="time step t = "), steps=[
        dict(label=str(t), method="animate",
             args=[[str(t)], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        for t in range(N + 1)])],
)
fig.show()

# --- Inputs over time ---
time = dt * np.arange(N)
fig_u = go.Figure()
fig_u.add_trace(go.Scatter(x=time, y=u.value[0], mode="lines+markers", name="u_t, first coordinate"))
fig_u.add_trace(go.Scatter(x=time, y=u.value[1], mode="lines+markers", name="u_t, second coordinate"))
for s in (1, -1):
    fig_u.add_hline(y=s * u_max, line_dash="dash", line_color="gray")
fig_u.update_layout(height=320, xaxis_title="time [s]", yaxis_title="acceleration input")
fig_u.show()

---

<font color="#A890FF"><b>✏️ TO EXPLORE</b></font>

Predict first, then change the data and re-run the two cells above:
- move the doorway (change `gate_y`), or the crossing time (change `t_gate`, e.g. to 15 or 25);
- add a speed limit $|v_{t,i}| \le 1.2$;
- halve the horizon $N$ while doubling $\Delta t$, so that the total time $N \Delta t$ stays the same. Does the control effort change? Why?

<font color="#CC79A7"><b>✨ SOLUTION</b></font>

Moving the crossing time far from $N/2$ makes the problem infeasible with $u_{\max} = 1$ (roughly outside $14 \le t_{\text{gate}} \le 27$): the robot cannot cover the distance in time. Without a speed limit, each velocity coordinate reaches about $1.5$ in absolute value; with $|v_{t,i}| \le 1.2$ the robot cruises at $1.2$ for longer and the control effort rises to about $11.1$. With $N = 20$ and $\Delta t = 0.5$ the control effort drops to about $4.87$, because the cost adds up fewer samples. The time-weighted effort $\Delta t \sum_t \|u_t\|_2^2$, an approximation of $\int \|u(\tau)\|_2^2 \, d\tau$, stays about $2.4$ in both cases.

The two update equations are a state-space model $x_{t+1} = A x_t + B u_t$ with state $x_t = (p_t, v_t)$, the topic of **Tutorial 2**. In **Tutorial 4** (MPC) we re-solve planning problems like this one at every time step from the measured state, which lets the controller react to disturbances.